# XCT Membrane Segmentation — U-Net (segmentation_models / Keras)

Trains a U-Net on the 15 hand-corrected ground-truth slices from the
annotation notebook, then runs it on the other 35 sampled slices that don't
have ground truth yet. Reuses the same `output_dir` layout (`images/`,
`masks_corrected/`, `sample_manifest.csv`) — run this after the annotation
notebook, pointed at the same output folder.

**Why patches, not full images:** slices are ~2028x2028 and there are only
15 labelled ones — training on small random crops (patches) instead of full
images does two things at once: keeps memory/compute manageable, and turns
15 images into effectively thousands of distinct training crops per epoch
via random position + augmentation. Inference stitches patch predictions
back together over a sliding window.

**Known compatibility gotcha:** `segmentation_models` hasn't been updated
for Keras 3 / TensorFlow >= 2.16. If you're on a recent TensorFlow and hit
import errors, either pin `tensorflow<2.16`, or install `tf-keras` and set
`TF_USE_LEGACY_KERAS=1` before importing TensorFlow.


## 0. Setup

In [ ]:
# pip install "tensorflow<2.16" segmentation-models albumentations scikit-learn tifffile tqdm

import os
os.environ["SM_FRAMEWORK"] = "tf.keras"  # must be set BEFORE importing segmentation_models

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from tqdm.auto import tqdm

import tensorflow as tf
import segmentation_models as sm
import albumentations as A
from sklearn.model_selection import train_test_split
from skimage.color import label2rgb

print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))


## 1. Config

In [ ]:
CONFIG = {
    # same output_dir as the annotation notebook — reuses images/, masks_corrected/, sample_manifest.csv
    "output_dir": Path("./xct_segmentation_output"),
    "model_dir": Path("./xct_unet_model"),

    "phase_names": ["voidage", "membrane", "polymer_cartridge"],
    "n_classes": 3,

    # --- model ---
    "backbone": "resnet34",       # imagenet-pretrained encoder; needs 3-channel input
    "patch_size": 256,
    "learning_rate": 1e-4,

    # --- training ---
    "batch_size": 8,
    "patches_per_epoch": 320,     # random patches sampled per epoch (train_steps = this / batch_size)
    "val_tiles_per_image": 9,     # deterministic grid tiles per validation slice (fixed, not resampled each epoch)
    "val_fraction": 0.2,          # fraction of the 15 corrected slices held out for validation
    "epochs": 60,
    "random_seed": 42,

    # --- inference ---
    "inference_stride_fraction": 0.5,  # sliding-window stride as a fraction of patch_size (0.5 = 50% overlap)
}

CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)
(CONFIG["output_dir"] / "masks_unet").mkdir(exist_ok=True)
CONFIG


## 2. Load the manifest, split labelled vs. unlabelled

`corrected == True` rows are your 15 hand-verified ground-truth slices
(training data). Everything else is what this notebook will predict on.


In [ ]:
manifest_path = CONFIG["output_dir"] / "sample_manifest.csv"
manifest = pd.read_csv(manifest_path)

def sample_filename(row):
    return f"sample_{row.sample_id:03d}_{Path(row.filename).stem}.tif"

labelled = manifest.loc[manifest.corrected].reset_index(drop=True)
unlabelled = manifest.loc[~manifest.corrected].reset_index(drop=True)

print(f"{len(labelled)} ground-truth (corrected) slices, {len(unlabelled)} remaining to segment")
assert len(labelled) >= 4, "Need at least a handful of corrected slices before training — check output_dir."

train_rows, val_rows = train_test_split(
    labelled, test_size=CONFIG["val_fraction"], random_state=CONFIG["random_seed"]
)
print(f"{len(train_rows)} training slices, {len(val_rows)} validation slices")


## 3. Intensity normalization

XCT slices are raw attenuation values, not 0-255 images — rescale using
percentiles computed from the training slices only (avoids leaking
validation/inference statistics into the normalization), then replicate to
3 channels since the pretrained encoder expects RGB-shaped input.


In [ ]:
def load_image(fname):
    return tifffile.imread(CONFIG["output_dir"] / "images" / fname).astype(np.float32)

def load_mask(fname):
    return tifffile.imread(CONFIG["output_dir"] / "masks_corrected" / fname)

_sample_imgs = [load_image(sample_filename(row)) for _, row in train_rows.iterrows()]
_all_vals = np.concatenate([im.ravel() for im in _sample_imgs])
P_LOW, P_HIGH = np.percentile(_all_vals, [1, 99])
print(f"Normalizing with 1st/99th percentiles from training slices: {P_LOW:.1f} / {P_HIGH:.1f}")
del _sample_imgs, _all_vals

def normalize_to_uint8(img):
    clipped = np.clip(img, P_LOW, P_HIGH)
    scaled = (clipped - P_LOW) / (P_HIGH - P_LOW + 1e-8)
    return (scaled * 255).astype(np.uint8)

def to_rgb(img_uint8):
    return np.stack([img_uint8] * 3, axis=-1)

def one_hot(mask):
    return np.stack([(mask == c) for c in range(CONFIG["n_classes"])], axis=-1).astype(np.float32)

preprocess_input = sm.get_preprocessing(CONFIG["backbone"])


## 4. Training data: random augmented patches

`PatchGenerator` preloads all 12-ish training images/masks into memory once
(small enough at this count), then serves random augmented crops each batch.
Augmentation matters a lot here given how few labelled slices there are —
flips/rotations/brightness jitter effectively multiply the training set.


In [ ]:
train_transform = A.Compose([
    A.RandomCrop(CONFIG["patch_size"], CONFIG["patch_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])

class PatchGenerator(tf.keras.utils.Sequence):
    def __init__(self, rows, batch_size, steps, patch_size):
        self.rows = rows.reset_index(drop=True)
        self.batch_size = batch_size
        self.steps = steps
        self.patch_size = patch_size
        self.rng = np.random.default_rng(CONFIG["random_seed"])

        self.images, self.masks = {}, {}
        for _, row in self.rows.iterrows():
            fname = sample_filename(row)
            self.images[row.sample_id] = normalize_to_uint8(load_image(fname))
            self.masks[row.sample_id] = load_mask(fname)

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        batch_imgs, batch_masks = [], []
        for _ in range(self.batch_size):
            sid = self.rng.choice(self.rows["sample_id"].values)
            img_rgb = to_rgb(self.images[sid])
            mask = self.masks[sid]

            aug = train_transform(image=img_rgb, mask=mask)
            patch_img = preprocess_input(aug["image"].astype(np.float32))
            patch_mask = one_hot(aug["mask"])

            batch_imgs.append(patch_img)
            batch_masks.append(patch_mask)

        return np.stack(batch_imgs), np.stack(batch_masks)

train_steps = max(1, CONFIG["patches_per_epoch"] // CONFIG["batch_size"])
train_generator = PatchGenerator(train_rows, CONFIG["batch_size"], train_steps, CONFIG["patch_size"])
print(f"{train_steps} steps/epoch, {CONFIG['batch_size']} patches/step")


## 5. Validation data: fixed grid tiles

Unlike training, validation uses the same fixed set of tiles every epoch
(a small grid per slice, no randomness) so `val_loss`/`val_iou_score` are
directly comparable across epochs instead of jumping around from resampling.


In [ ]:
def extract_grid_tiles(img_rgb, mask, patch_size, n_tiles):
    H, W = mask.shape
    n_side = int(np.ceil(np.sqrt(n_tiles)))
    ys = np.linspace(0, max(H - patch_size, 0), n_side).astype(int)
    xs = np.linspace(0, max(W - patch_size, 0), n_side).astype(int)
    tiles = []
    for y in ys:
        for x in xs:
            if len(tiles) >= n_tiles:
                break
            tiles.append((img_rgb[y:y + patch_size, x:x + patch_size],
                          mask[y:y + patch_size, x:x + patch_size]))
    return tiles

X_val, Y_val = [], []
for _, row in val_rows.iterrows():
    fname = sample_filename(row)
    img_rgb = to_rgb(normalize_to_uint8(load_image(fname)))
    mask = load_mask(fname)
    for patch_img, patch_mask in extract_grid_tiles(img_rgb, mask, CONFIG["patch_size"], CONFIG["val_tiles_per_image"]):
        X_val.append(preprocess_input(patch_img.astype(np.float32)))
        Y_val.append(one_hot(patch_mask))

X_val = np.stack(X_val)
Y_val = np.stack(Y_val)
print(f"Validation set: {X_val.shape[0]} tiles from {len(val_rows)} slice(s)")


## 6. Build and compile the U-Net

Dice + categorical focal loss handles the class imbalance reasonably well
(voidage dominates the pixel count; membrane and polymer are minority
classes) without needing manually-tuned class weights.


In [ ]:
model = sm.Unet(
    CONFIG["backbone"],
    classes=CONFIG["n_classes"],
    activation="softmax",
    encoder_weights="imagenet",
    input_shape=(CONFIG["patch_size"], CONFIG["patch_size"], 3),
)

total_loss = sm.losses.DiceLoss() + sm.losses.CategoricalFocalLoss()

model.compile(
    optimizer=tf.keras.optimizers.Adam(CONFIG["learning_rate"]),
    loss=total_loss,
    metrics=[sm.metrics.IOUScore(threshold=0.5), sm.metrics.FScore(threshold=0.5)],
)

model.summary()


## 7. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(CONFIG["model_dir"] / "best_model.h5"),
        save_best_only=True, monitor="val_iou_score", mode="max",
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_iou_score", mode="max", factor=0.5, patience=5),
    tf.keras.callbacks.EarlyStopping(monitor="val_iou_score", mode="max", patience=12, restore_best_weights=True),
]

history = model.fit(
    train_generator,
    validation_data=(X_val, Y_val),
    epochs=CONFIG["epochs"],
    callbacks=callbacks,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["iou_score"], label="train")
axes[1].plot(history.history["val_iou_score"], label="val")
axes[1].set_title("IoU score"); axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
model.save(str(CONFIG["model_dir"] / "final_model.h5"))
print("Saved model ->", CONFIG["model_dir"] / "final_model.h5")


## 8. Tiled inference on full-resolution slices

Slides the trained patch-sized model over a full slice with overlap,
averaging overlapping softmax predictions before taking the final class —
this avoids the harsh tile-boundary artifacts a non-overlapping grid would
produce.


In [ ]:
def predict_full_image(model, img_raw, patch_size=None, stride=None):
    patch_size = patch_size or CONFIG["patch_size"]
    stride = stride or max(1, int(patch_size * CONFIG["inference_stride_fraction"]))

    img_rgb = to_rgb(normalize_to_uint8(img_raw))
    H, W = img_raw.shape

    pad_h = (patch_size - H % stride) % stride
    pad_w = (patch_size - W % stride) % stride
    img_padded = np.pad(img_rgb, ((0, pad_h + patch_size), (0, pad_w + patch_size), (0, 0)), mode="reflect")
    Hp, Wp = img_padded.shape[:2]

    ys = list(range(0, Hp - patch_size + 1, stride))
    xs = list(range(0, Wp - patch_size + 1, stride))

    batch, positions = [], []
    for y in ys:
        for x in xs:
            tile = img_padded[y:y + patch_size, x:x + patch_size]
            batch.append(preprocess_input(tile.astype(np.float32)))
            positions.append((y, x))
    batch = np.stack(batch)

    preds = model.predict(batch, batch_size=CONFIG["batch_size"], verbose=0)

    prob_sum = np.zeros((Hp, Wp, CONFIG["n_classes"]), dtype=np.float32)
    weight = np.zeros((Hp, Wp), dtype=np.float32)
    for (y, x), pred in zip(positions, preds):
        prob_sum[y:y + patch_size, x:x + patch_size] += pred
        weight[y:y + patch_size, x:x + patch_size] += 1

    prob_avg = prob_sum[:H, :W] / np.maximum(weight[:H, :W, None], 1e-8)
    return np.argmax(prob_avg, axis=-1).astype(np.uint8)

def overlay(img, labels):
    img_norm = (img - img.min()) / max(1, (img.max() - img.min()))
    return label2rgb(labels.astype(np.int32), image=img_norm, bg_label=-1, alpha=0.35)


In [ ]:
unet_output_dir = CONFIG["output_dir"] / "masks_unet"

for _, row in tqdm(list(unlabelled.iterrows()), desc="U-Net inference"):
    fname = sample_filename(row)
    img_raw = load_image(fname)
    pred_mask = predict_full_image(model, img_raw)
    tifffile.imwrite(unet_output_dir / fname, pred_mask)

print(f"Saved {len(unlabelled)} predicted masks -> {unet_output_dir}")


## 9. QC: spot-check predictions on the unlabelled slices

In [ ]:
n_preview = 6
preview_rows = unlabelled.sample(n=min(n_preview, len(unlabelled)), random_state=CONFIG["random_seed"])

fig, axes = plt.subplots(2, len(preview_rows), figsize=(3 * len(preview_rows), 6))
for col, (_, row) in enumerate(preview_rows.iterrows()):
    fname = sample_filename(row)
    img = load_image(fname)
    pred = tifffile.imread(unet_output_dir / fname)

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"sample {row.sample_id}", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(overlay(img, pred))
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("raw")
axes[1, 0].set_ylabel("U-Net prediction")
plt.tight_layout()
plt.show()


## 10. Next steps

- If the predictions on the 35 unlabelled samples look reasonable, the same
  `predict_full_image()` function can be run across the full ~4000-slice
  stack — loop over pages lazily (as in the annotation notebook's
  `tif_file.pages[idx].asarray()`) rather than loading everything at once,
  and write each prediction out as you go rather than holding the whole
  volume in memory.
- If predictions look weak in specific regions, hand-correcting a few more
  of the remaining 35 (using the annotation notebook) and re-running this
  notebook is usually more effective than tuning hyperparameters — more
  varied ground truth tends to matter more than model tweaks at this scale
  of dataset.
- `CONFIG["model_dir"] / "best_model.h5"` holds the best checkpoint by
  validation IoU if you want to reload it later without retraining:
  `model = tf.keras.models.load_model(path, custom_objects={...})` — needs
  the same custom loss/metric objects from `segmentation_models` passed in
  via `custom_objects`, or reconstruct the architecture and call
  `model.load_weights(path)` instead, which sidesteps that entirely.
